# Sequential Multi-Agent Workflow with Microsoft Agent Framework

## Overview

This notebook demonstrates a **two-agent sequential workflow** using the **Microsoft Agent Framework** for Python:

| Agent | Role |
|-------|------|
| **Agent 1 — Itinerary Planner** | Receives a travel prompt and produces a detailed day-by-day itinerary with named stops |
| **Agent 2 — Todo List Creator** | Reads the itinerary and converts each stop into a structured todo-list output |

### Architecture

```
User Prompt
     │
     ▼
┌──────────────────────────────┐
│   Agent 1                    │
│   ItineraryPlanner           │
│   — Plans day-by-day stops   │
└──────────────┬───────────────┘
               │  conversation history passed forward
               ▼
┌──────────────────────────────┐
│   Agent 2                    │
│   TravelTodoCreator          │
│   — Extracts stops as todos  │
│   — Outputs structured list  │
└──────────────────────────────┘
               │
               ▼
          Todo List Output ✓
```

### Key Concepts Demonstrated
- **Agent definition**: `Agent` + `AzureOpenAIResponsesClient` — orchestration logic runs in-process, inference via Azure AI Responses API
- **Sequential orchestration**: `SequentialBuilder` from `agent_framework_orchestrations` — chains agents and passes conversation context between them
- **Event streaming**: Live event feed from the workflow runtime including executor lifecycle, data events, and final output


## 1. Install Required Dependencies

The following packages power this notebook:

| Package | Purpose |
|---------|---------|
| `agent-framework` | Core agent primitives (`Agent`, `Message`, `WorkflowBuilder`) |
| `agent-framework-azure-ai` | Azure AI provider: `AzureOpenAIResponsesClient` |
| `agent-framework-orchestrations` | `SequentialBuilder` for chaining agents |
| `azure-identity` | `DefaultAzureCredential` for passwordless auth |
| `python-dotenv` | Load `.env` configuration |


In [ ]:
import subprocess, sys

packages = [
    "agent-framework",
    "agent_framework.azure",
    "agent-framework-azure-ai",
    "agent-framework-orchestrations",
    "azure-identity",
    "python-dotenv",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", *packages],
    capture_output=True,
    text=True,
)
if result.returncode == 0:
    print("✅ All packages ready.")
else:
    print("⚠️ Install warnings/errors:")
    print(result.stderr[-2000:])


## 2. Import Libraries and Configure Environment

We load configuration from the `.env` file in the same directory. The key variable is:

- `AZURE_AI_PROJECT_ENDPOINT` — The Azure AI Foundry project URL  
  (e.g. `https://msf-hacktest.services.ai.azure.com/api/projects/proj-default`)
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` *(optional)* — Model to use (defaults to `gpt-4o`)

Authentication uses `DefaultAzureCredential`, which picks up Azure CLI login, managed identity, or environment-provided service principal credentials automatically.


In [ ]:
import os
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from azure.identity.aio import AzureCliCredential

from agent_framework import Agent, Message
from agent_framework.azure import AzureOpenAIResponsesClient
from agent_framework_orchestrations import SequentialBuilder

from IPython.display import display, Markdown, HTML

# ── Load .env from the same directory as this notebook ──────────────────────
env_path = Path(__file__).parent / ".env" if "__file__" in dir() else Path(".env")
load_dotenv(dotenv_path=env_path, override=False)

# ── Configuration ────────────────────────────────────────────────────────────
PROJECT_ENDPOINT = os.environ.get("AZURE_AI_PROJECT_ENDPOINT", "https://msf-hacktest.services.ai.azure.com/api/projects/proj-default")
MODEL_DEPLOYMENT = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-5")

# Shared credential — uses Azure CLI login (ensures correct tenant)
credential = AzureCliCredential()

# ── Validation ───────────────────────────────────────────────────────────────
if not PROJECT_ENDPOINT:
    raise EnvironmentError(
        "AZURE_AI_PROJECT_ENDPOINT is not set.\n"
        "Add it to your .env or set it as an environment variable:\n"
        '  AZURE_AI_PROJECT_ENDPOINT="https://<resource>.services.ai.azure.com/api/projects/<project>"'
    )

print("✅ Configuration loaded")
print(f"   Project endpoint : {PROJECT_ENDPOINT}")
print(f"   Model deployment : {MODEL_DEPLOYMENT}")


## 3. Agent 1 — Itinerary Planner

Agent 1 uses `AzureOpenAIResponsesClient` which connects to the Azure AI Project Responses API for inference. The orchestration loop runs locally inside this notebook.

**Role:** Receive a travel destination prompt and produce a structured, day-by-day itinerary with clearly named stops. The output is structured so Agent 2 can parse each stop and create a corresponding todo item.

```
User Prompt ──► Agent 1 ──► Structured Itinerary
```


In [ ]:
ITINERARY_AGENT_INSTRUCTIONS = """\
You are an expert travel planner. When given a travel destination and duration, you create
a concise, well-structured day-by-day itinerary.

Output format — strictly follow this structure:
1. A brief intro paragraph (2 sentences max).
2. A numbered list of stops/activities — one per line, prefixed with "STOP:" — for example:
   STOP: Day 1 Morning — Arrive at Narita Airport and check in to Shinjuku hotel
   STOP: Day 1 Afternoon — Explore Shinjuku Gyoen National Garden
   STOP: Day 1 Evening — Dinner at Omoide Yokocho (Memory Lane)
   ...
3. A short closing tip (1 sentence).

Rules:
- Every scheduled activity must be on its own "STOP:" line.
- Keep each stop description under 15 words.
- Include at least 3 stops per day.
- Do not use bullet points or markdown headers — plain numbered STOP lines only.
"""

local_client = AzureOpenAIResponsesClient(
    project_endpoint=PROJECT_ENDPOINT,
    deployment_name=MODEL_DEPLOYMENT,
    credential=credential,
)

agent1 = Agent(
    client=local_client,
    name="ItineraryPlanner",
    instructions=ITINERARY_AGENT_INSTRUCTIONS,
    description="Creates structured travel itineraries with per-stop STOP: lines.",
)

print("✅ Agent 1 ready")
print(f"   Name : {agent1.name}")
print(f"   Model: {MODEL_DEPLOYMENT}")


## 4. Agent 2 — Todo List Creator

Agent 2 receives the full conversation (including Agent 1's itinerary) and converts each stop into a structured todo-list output.

```
Itinerary from Agent 1
        │
        ▼
  Agent 2 (TravelTodoCreator)
  — Parses STOP: lines
  — Outputs formatted todo list
```


In [ ]:
TODO_AGENT_INSTRUCTIONS = """\
You are a task organizer. You receive a travel itinerary from a colleague and your
job is to convert it into a clear, actionable todo list.

For each line that starts with "STOP:", extract the activity description and format
it as a numbered todo item with a checkbox.

Output format:
- Start with a title line: "## Travel Todo List"
- Each item on its own line: "[ ] 1. <activity description>"
- Group items by day with a blank line between days
- End with a summary line: "Total tasks: <count>"

If the itinerary has no STOP: lines, summarize the itinerary into logical stops
yourself and create todo items from those.
"""

agent2 = Agent(
    client=local_client,
    name="TravelTodoCreator",
    instructions=TODO_AGENT_INSTRUCTIONS,
    description="Converts a travel itinerary into a structured todo list.",
)

print("✅ Agent 2 ready")
print(f"   Name : {agent2.name}")
print(f"   Model: {MODEL_DEPLOYMENT}")


## 5. Build the Sequential Workflow

The `SequentialBuilder` from `agent_framework_orchestrations` chains agents in order, passing a shared `list[Message]` conversation context along the chain.

```
Input ──► Agent 1 (ItineraryPlanner) ──► Agent 2 (TravelTodoCreator) ──► Final Output
```

- Agent 1 receives the user prompt and generates the itinerary
- Agent 1's full conversation is forwarded to Agent 2
- Agent 2 sees the itinerary and produces a structured todo list


In [ ]:
# ── Build the sequential workflow ─────────────────────────────────────────────
workflow = SequentialBuilder(
    participants=[agent1, agent2],
).build()

print("✅ Sequential workflow built")
print(f"   Pipeline: {agent1.name} ──► {agent2.name}")
print(f"   Entry types  : {[t.__name__ for t in workflow.input_types]}")
print(f"   Output types : {[t.__name__ for t in workflow.output_types]}")


## 6. Execute the Workflow

The cell below:
1. Launches the workflow with a sample travel prompt using **streaming mode** (`stream=True`)
2. Captures intermediate data from each agent as it completes
3. Renders **Agent 1's itinerary** and **Agent 2's todo list** as rich Markdown output


In [ ]:
# ── Travel prompt to process ──────────────────────────────────────────────────
USER_PROMPT = "Plan a 3-day trip to Switzerland focusing on Zurich and Lucerne."

display(Markdown(f"## 🗺️ Sequential Workflow\n\n**Prompt:** *{USER_PROMPT}*\n\n---"))

# ── Run with streaming ───────────────────────────────────────────────────────
stream = workflow.run(USER_PROMPT, stream=True)

# Track which executors have completed so we can label Agent 1 vs Agent 2
completed_executors = []
agent_outputs = {}  # executor_id -> list of messages

async for event in stream:
    etype = event.type
    ts = datetime.now().strftime("%H:%M:%S")

    if etype == "started":
        print(f"[{ts}] 🟢 Workflow started")

    elif etype == "executor_invoked":
        executor_id = getattr(event, "executor_id", "?")
        print(f"[{ts}] 🚀 Running: {executor_id}")

    elif etype == "executor_completed":
        executor_id = getattr(event, "executor_id", "?")
        completed_executors.append(executor_id)
        print(f"[{ts}] ✅ Completed: {executor_id}")

    elif etype == "data":
        # Capture conversation data from each executor
        executor_id = getattr(event, "executor_id", "?")
        if isinstance(event.data, list):
            agent_outputs[executor_id] = event.data

    elif etype == "failed" or etype == "executor_failed":
        details = getattr(event, "details", event.data)
        print(f"[{ts}] ❌ Failed: {details}")

print(f"\n✅ Workflow complete — {len(completed_executors)} executors ran")

# ── Extract final result ─────────────────────────────────────────────────────
result = await stream.get_final_response()
outputs = result.get_outputs()

# ── Helper: extract assistant text from a message list ────────────────────────
def get_assistant_text(messages):
    """Return the last assistant message text from a list of Message objects."""
    for msg in reversed(messages):
        role = msg.role if hasattr(msg, "role") else ""
        text = msg.text if hasattr(msg, "text") else str(msg)
        if role == "assistant" and text:
            return text
    return None

# ── Display Agent 1 output (Itinerary) ──────────────────────────────────────
# Agent 1's output is captured in agent_outputs from the data events.
# If not captured there, fall back to extracting from the final conversation.
itinerary_text = None
todo_text = None

# Try to get from captured data events (keyed by executor_id)
executor_ids = list(agent_outputs.keys())
if len(executor_ids) >= 1:
    itinerary_text = get_assistant_text(agent_outputs[executor_ids[0]])
if len(executor_ids) >= 2:
    todo_text = get_assistant_text(agent_outputs[executor_ids[-1]])

# Fallback: extract from final outputs (the full conversation from Agent 2)
if not itinerary_text or not todo_text:
    if outputs:
        final_conversation = outputs[-1] if isinstance(outputs[-1], list) else outputs
        if isinstance(final_conversation, list):
            assistant_msgs = [
                (msg.text if hasattr(msg, "text") else str(msg))
                for msg in final_conversation
                if (hasattr(msg, "role") and msg.role == "assistant" and
                    (msg.text if hasattr(msg, "text") else None))
            ]
            if len(assistant_msgs) >= 2:
                itinerary_text = itinerary_text or assistant_msgs[0]
                todo_text = todo_text or assistant_msgs[-1]
            elif len(assistant_msgs) == 1:
                todo_text = todo_text or assistant_msgs[0]

# ── Render outputs ───────────────────────────────────────────────────────────
if itinerary_text:
    display(Markdown("---\n## 🗓️ Agent 1 — Itinerary\n"))
    display(Markdown(itinerary_text))

if todo_text:
    display(Markdown("---\n## ✅ Agent 2 — Todo List\n"))
    display(Markdown(todo_text))

if not itinerary_text and not todo_text:
    print("⚠️ No agent outputs captured.")
    if outputs:
        for output in outputs:
            if isinstance(output, list):
                for msg in output:
                    text = msg.text if hasattr(msg, "text") else str(msg)
                    if text:
                        print(text)

display(Markdown("---\n*Workflow complete.*"))


## 7. Cleanup

Close the credential to release resources. This cell is optional — resources are also released when the kernel shuts down.


In [ ]:
await credential.close()
print("✅ Resources released.")
